# 02 — Exploratory Data Analysis
**FootScout — BHT Berlin**

EDA with reusable radar chart, distributions, correlations, World Cup spotlight.


In [ ]:
import sys; sys.path.insert(0,"..")
import pandas as pd, numpy as np, plotly.express as px, plotly.graph_objects as go
from src.recommender import get_radar_data, make_radar_figure
df = pd.read_csv("../data/processed/players_merged.csv", low_memory=False)
print(f"{len(df):,} players loaded")

## 1. Player Distribution by League & Position

In [ ]:
fig = px.histogram(df, x="league", color="pos", barmode="group",
    title="Player Distribution", template="plotly_dark"); fig.show()

## 2. ★ Radar Chart — Reusable Utility (also used in Streamlit)

In [ ]:
# This exact function is imported into app/streamlit_app.py
sample = df["player"].iloc[0]
radar = get_radar_data(sample, df, position_avg=True)
fig = make_radar_figure(radar, show_average=True)
fig.show()

## 3. xG/90 vs Goals/90 — Over/Under-Performers

In [ ]:
if all(c in df.columns for c in ["xg_per90","gls_per90"]):
    fig = px.scatter(df, x="xg_per90", y="gls_per90", color="league",
        hover_name="player", trendline="ols",
        title="xG/90 vs Goals/90", template="plotly_dark")
    fig.add_shape(type="line",x0=0,y0=0,x1=1,y1=1,line=dict(dash="dot",color="white"))
    fig.show()

## 4. Correlation Heatmap

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
STAT_COLS=[c for c in df.columns if "_per90" in c or c=="pass_completion_pct"]
corr=df[STAT_COLS].apply(pd.to_numeric,errors="coerce").corr()
fig,ax=plt.subplots(figsize=(10,8)); sns.heatmap(corr,cmap="coolwarm",center=0,ax=ax,annot=False)
ax.set_title("Feature Correlation Heatmap"); plt.tight_layout(); plt.show()

## 5. Market Value Distribution

In [ ]:
if "market_value_eur" in df.columns:
    fig=px.histogram(df,x="market_value_eur",log_x=True,color="league",nbins=50,
        title="Market Value (log scale)",template="plotly_dark",opacity=0.7)
    fig.show()